In [1]:
import torch
import math
from torch_geometric.data import Data

# ===================================================================
# MOORING LINE PROPERTIES  (Santjer et al. 2025 – Table 2)
# ===================================================================
MOORING_PROPERTIES = {
    "diameter":         0.068,        # [m]
    "area":             0.003619,     # [m2]
    "mass_per_length":  28.23,        # [kg/m]  (rhoA)
    "EA":               232.7e6,      # [N]     axial stiffness
    "Cd_n":             2.6,          # normal drag coefficient
    "Cd_t":             1.4,          # axial drag coefficient
    "k_b":              3.0e6,        # [Pa/m]  seabed spring stiffness
    "c_b":              3.0e5,        # [Pa*s/m] seabed damping
}

# ===================================================================
# LOCATION CONFIGURATIONS  (Santjer et al. 2025 – Tables 1 & 3)
# ===================================================================
# h              - total water depth [m]
# L0             - unstretched line length [m]
# L_susp         - suspended catenary length [m]
# X_AN           - anchor position [x_m, z_m],  z negative (below MSL)
# X_FL           - fairlead position [x_m, z_m], z = -3 m for all
# depths_current - 5 depths [m above seabed] for current velocity profiles
LOCATION_CONFIGS = {
    1:  {"h": 12.6, "L0":  75, "L_susp": 10.72,
         "X_AN": [0.0, -12.6], "X_FL": [67.7, -3.0],
         "depths_current": [1, 4,  6,  9, 12]},
    2:  {"h": 14.5, "L0":  75, "L_susp": 12.79,
         "X_AN": [0.0, -14.5], "X_FL": [66.3, -3.0],
         "depths_current": [1, 4,  7, 10, 14]},
    3:  {"h": 15.0, "L0":  75, "L_susp": 13.40,
         "X_AN": [0.0, -15.0], "X_FL": [65.9, -3.0],
         "depths_current": [1, 4,  7, 11, 14]},
    4:  {"h": 22.8, "L0":  75, "L_susp": 22.14,
         "X_AN": [0.0, -22.8], "X_FL": [60.0, -3.0],
         "depths_current": [1, 6, 11, 17, 22]},
    5:  {"h": 30.1, "L0":  75, "L_susp": 30.27,
         "X_AN": [0.0, -30.1], "X_FL": [54.5, -3.0],
         "depths_current": [1, 8, 15, 22, 29]},
    6:  {"h": 30.2, "L0":  75, "L_susp": 30.36,
         "X_AN": [0.0, -30.2], "X_FL": [54.4, -3.0],
         "depths_current": [1, 8, 15, 22, 29]},
    7:  {"h": 33.6, "L0":  75, "L_susp": 34.16,
         "X_AN": [0.0, -33.6], "X_FL": [51.8, -3.0],
         "depths_current": [1, 9, 17, 25, 33]},
    8:  {"h": 35.7, "L0":  75, "L_susp": 36.46,
         "X_AN": [0.0, -35.7], "X_FL": [50.2, -3.0],
         "depths_current": [1, 9, 18, 26, 35]},
    9:  {"h": 38.6, "L0": 105, "L_susp": 39.78,
         "X_AN": [0.0, -38.6], "X_FL": [78.0, -3.0],
         "depths_current": [1, 10, 19, 28, 38]},
    10: {"h": 39.2, "L0": 105, "L_susp": 40.43,
         "X_AN": [0.0, -39.2], "X_FL": [77.6, -3.0],
         "depths_current": [1, 10, 20, 29, 38]},
    11: {"h": 40.7, "L0": 105, "L_susp": 42.15,
         "X_AN": [0.0, -40.7], "X_FL": [76.4, -3.0],
         "depths_current": [1, 11, 20, 30, 40]},
    12: {"h": 44.7, "L0": 105, "L_susp": 46.54,
         "X_AN": [0.0, -44.7], "X_FL": [73.4, -3.0],
         "depths_current": [1, 12, 22, 33, 44]},
}

# Environmental feature names (inputs from hydrodynamic model / GCBN)
ENV_FEATURE_NAMES = ["Hs", "Tp", "cd1", "cd2", "cd3", "cd4", "cd5"]


def build_mooring_graph(location_id: int, num_intermediate: int = 19) -> Data:
    """
    Build a static PyG graph for one of the 12 North Sea study locations.

    Nodes (num_intermediate+2): anchor r0, num_intermediate intermediate nodes, fairlead rN.
    All material properties match Table 2 of Santjer et al. (2025).
    Node positions are linearly interpolated between anchor and fairlead
    along the suspended length; the FE solver resolves the true catenary.

    Static node features [N x 15]:
        0  s_over_L_susp    normalised arc-length position [0..1]
        1  is_anchor        1 for r0
        2  is_fairlead      1 for r5
        3  is_intermediate  1 for r1-r4
        4  node_type_id     0=anchor, 1=intermediate, 2=fairlead
        5  x0               reference x-coordinate [m]
        6  z0               reference z-coordinate [m]
        7  nodal_mass       lumped mass [kg]
        8  diameter         line diameter [m]
        9  area             cross-sectional area [m2]
        10 submerged_weight submerged weight contribution [N]
        11 z_bed            seabed elevation [m]  (= -water_depth)
        12 can_touch_seabed 1 only for anchor node (r0)
        13 Cd_n             normal drag coefficient
        14 Cd_t             axial drag coefficient

    Static edge features [E x 8]:
        0  segment_length             reference arc length [m]
        1  segment_length_over_L_susp normalised segment length
        2  EA                         axial stiffness [N]
        3  mass_per_length            linear density [kg/m]
        4  diameter                   line diameter [m]
        5  tangent_x                  x-component of reference tangent
        6  tangent_z                  z-component of reference tangent
        7  edge_type_id               0 for all mooring-line segments

    Parameters
    ----------
    location_id : int
        Study location (1-12) from Santjer et al. (2025).
    num_intermediate : int
        Intermediate nodes between anchor and fairlead (default 4 -> 6 nodes).
    """
    if location_id not in LOCATION_CONFIGS:
        raise ValueError(
            f"location_id must be one of {sorted(LOCATION_CONFIGS.keys())}, "
            f"got {location_id}."
        )

    cfg   = LOCATION_CONFIGS[location_id]
    props = MOORING_PROPERTIES

    h      = cfg["h"]
    L0     = cfg["L0"]
    L_susp = cfg["L_susp"]
    x_an   = float(cfg["X_AN"][0])
    z_an   = float(cfg["X_AN"][1])
    x_fl   = float(cfg["X_FL"][0])
    z_fl   = float(cfg["X_FL"][1])

    # ------------------------------------------------------------------
    # 1. NODE POSITIONS  (linear interpolation along the catenary chord)
    # ------------------------------------------------------------------
    num_nodes = 2 + num_intermediate      # default: 6
    t_vals = torch.linspace(0.0, 1.0, num_nodes)
    x0 = x_an + t_vals * (x_fl - x_an)
    z0 = z_an + t_vals * (z_fl - z_an)

    # ------------------------------------------------------------------
    # 2. ARC-LENGTH COORDINATE along the suspended segment
    # ------------------------------------------------------------------
    s        = torch.linspace(0.0, L_susp, num_nodes)
    s_over_L = s / L_susp

    # ------------------------------------------------------------------
    # 3. NODE TYPE FLAGS
    # ------------------------------------------------------------------
    is_anchor       = torch.zeros(num_nodes)
    is_fairlead     = torch.zeros(num_nodes)
    is_intermediate = torch.ones(num_nodes)
    is_anchor[0]        = 1.0
    is_fairlead[-1]     = 1.0
    is_intermediate[0]  = 0.0
    is_intermediate[-1] = 0.0

    node_type_id      = torch.ones(num_nodes)
    node_type_id[0]   = 0.0    # anchor
    node_type_id[-1]  = 2.0    # fairlead

    # ------------------------------------------------------------------
    # 4. PHYSICAL NODE ATTRIBUTES
    # ------------------------------------------------------------------
    seg_len_nom = L_susp / (num_nodes - 1)

    nodal_mass = torch.full((num_nodes,), props["mass_per_length"] * seg_len_nom)
    nodal_mass[0]  *= 0.5
    nodal_mass[-1] *= 0.5

    diameter_node = torch.full((num_nodes,), props["diameter"])
    area_node     = torch.full((num_nodes,), props["area"])

    rho_water = 1025.0
    g         = 9.81
    submerged_weight_per_length = (
        props["mass_per_length"] - rho_water * props["area"]
    ) * g
    submerged_weight_node = torch.full(
        (num_nodes,), submerged_weight_per_length * seg_len_nom
    )
    submerged_weight_node[0]  *= 0.5
    submerged_weight_node[-1] *= 0.5

    z_bed      = z_an               # seabed level = anchor elevation
    z_bed_node = torch.full((num_nodes,), z_bed)

    can_touch_seabed    = torch.zeros(num_nodes)
    can_touch_seabed[0] = 1.0      # only the anchor node rests on the seabed

    Cd_n_node = torch.full((num_nodes,), props["Cd_n"])
    Cd_t_node = torch.full((num_nodes,), props["Cd_t"])

    # ------------------------------------------------------------------
    # 5. STATIC NODE FEATURE MATRIX  [N x 15]
    # ------------------------------------------------------------------
    x = torch.stack([
        s_over_L,              # 0
        is_anchor,             # 1
        is_fairlead,           # 2
        is_intermediate,       # 3
        node_type_id,          # 4
        x0,                    # 5
        z0,                    # 6
        nodal_mass,            # 7
        diameter_node,         # 8
        area_node,             # 9
        submerged_weight_node, # 10
        z_bed_node,            # 11
        can_touch_seabed,      # 12
        Cd_n_node,             # 13
        Cd_t_node,             # 14
    ], dim=1)

    # ------------------------------------------------------------------
    # 6. GRAPH CONNECTIVITY  –  bidirectional chain
    # ------------------------------------------------------------------
    senders, receivers = [], []
    for i in range(num_nodes - 1):
        senders   += [i,     i + 1]
        receivers += [i + 1, i    ]
    edge_index = torch.tensor([senders, receivers], dtype=torch.long)

    # ------------------------------------------------------------------
    # 7. STATIC EDGE FEATURES  [E x 8]  (same for forward and backward)
    # ------------------------------------------------------------------
    edge_features = []
    for i in range(num_nodes - 1):
        dx = float(x0[i + 1] - x0[i])
        dz = float(z0[i + 1] - z0[i])
        seg_len    = math.sqrt(dx ** 2 + dz ** 2)
        seg_normed = seg_len / L_susp
        tx = dx / seg_len if seg_len > 0 else 0.0
        tz = dz / seg_len if seg_len > 0 else 0.0
        feat = [
            seg_len,                   # 0
            seg_normed,                # 1
            props["EA"],               # 2
            props["mass_per_length"],  # 3
            props["diameter"],         # 4
            tx,                        # 5
            tz,                        # 6
            0.0,                       # 7  edge_type_id
        ]
        edge_features.extend([feat, feat])    # forward + backward

    edge_attr = torch.tensor(edge_features, dtype=torch.float32)

    # ------------------------------------------------------------------
    # 8. GRAPH OBJECT
    # ------------------------------------------------------------------
    pos  = torch.stack([x0, z0], dim=1)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos)

    data.location_id      = location_id
    data.num_nodes_total  = num_nodes
    data.water_depth      = h
    data.line_length      = L0
    data.suspended_length = L_susp
    data.ea               = props["EA"]
    data.mass_per_length  = props["mass_per_length"]
    data.diameter         = props["diameter"]
    data.area             = props["area"]
    data.k_b              = props["k_b"]
    data.c_b              = props["c_b"]
    data.z_bed            = z_bed
    data.z_bed_node       = z_bed_node
    rope_positions        = [f"r{i}" for i in range(num_nodes)]
    data.rope_positions   = rope_positions
    data.depths_current   = cfg["depths_current"]

    return data


def print_graph_summary(data: Data):
    print("----- GRAPH SUMMARY -----")
    print(f"Location ID     : {data.location_id}")
    print(f"Water depth [m] : {data.water_depth:.1f}")
    print(f"L0 [m]          : {data.line_length}")
    print(f"L_susp [m]      : {data.suspended_length:.2f}")
    print(f"EA [N]          : {data.ea:.3e}")
    print(f"Nodes           : {data.num_nodes_total}  ({', '.join(data.rope_positions)})")
    print()
    print(f"x shape       : {tuple(data.x.shape)}")
    print(f"edge_index    : {tuple(data.edge_index.shape)}")
    print(f"edge_attr     : {tuple(data.edge_attr.shape)}")
    print()
    print("Static node feature columns (15):")
    for i, c in enumerate([
        "s_over_L_susp", "is_anchor", "is_fairlead", "is_intermediate",
        "node_type_id", "x0", "z0", "nodal_mass", "diameter", "area",
        "submerged_weight", "z_bed", "can_touch_seabed", "Cd_n", "Cd_t",
    ]):
        print(f"  {i:2d}  {c}")
    print()
    print("Static edge feature columns (8):")
    for i, c in enumerate([
        "segment_length", "segment_length_over_L_susp", "EA",
        "mass_per_length", "diameter", "tangent_x", "tangent_z", "edge_type_id",
    ]):
        print(f"  {i}  {c}")
    print()
    print("Node positions (x, z) [m]:")
    print(data.pos)


if __name__ == "__main__":
    for loc_id in [4, 9, 12]:
        graph = build_mooring_graph(location_id=loc_id, num_intermediate=19)
        print_graph_summary(graph)
        print()


----- GRAPH SUMMARY -----
Location ID     : 4
Water depth [m] : 22.8
L0 [m]          : 75
L_susp [m]      : 22.14
EA [N]          : 2.327e+08
Nodes           : 21  (r0, r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13, r14, r15, r16, r17, r18, r19, r20)

x shape       : (21, 15)
edge_index    : (2, 40)
edge_attr     : (40, 8)

Static node feature columns (15):
   0  s_over_L_susp
   1  is_anchor
   2  is_fairlead
   3  is_intermediate
   4  node_type_id
   5  x0
   6  z0
   7  nodal_mass
   8  diameter
   9  area
  10  submerged_weight
  11  z_bed
  12  can_touch_seabed
  13  Cd_n
  14  Cd_t

Static edge feature columns (8):
  0  segment_length
  1  segment_length_over_L_susp
  2  EA
  3  mass_per_length
  4  diameter
  5  tangent_x
  6  tangent_z
  7  edge_type_id

Node positions (x, z) [m]:
tensor([[  0.0000, -22.8000],
        [  3.0000, -21.8100],
        [  6.0000, -20.8200],
        [  9.0000, -19.8300],
        [ 12.0000, -18.8400],
        [ 15.0000, -17.8500],
        [ 

In [ ]:
# -------------------------------------------------------------
# resampling utility
# -------------------------------------------------------------

import numpy as np


def resample_fe_output(data_tN: torch.Tensor, target_N: int) -> torch.Tensor:
    """
    Resample FE output from N_orig nodes to target_N nodes along the
    normalised arc-length axis using linear interpolation (np.interp).

    The anchor (node 0) and fairlead (node -1) are always preserved
    exactly because they are the endpoints of the np.linspace grid.

    Parameters
    ----------
    data_tN : torch.Tensor  shape [T, N_orig]
        Full FE output for a single signal (e.g. x_abs, z_abs, tension)
        at N_orig nodes.
    target_N : int
        Desired number of output nodes (includes anchor and fairlead).

    Returns
    -------
    torch.Tensor  shape [T, target_N], dtype=torch.float32
    """
    T, N_orig = data_tN.shape
    if target_N == N_orig:
        return data_tN.to(torch.float32)

    src_pos = np.linspace(0.0, 1.0, N_orig)
    tgt_pos = np.linspace(0.0, 1.0, target_N)

    arr = data_tN.numpy().astype(np.float64)
    out = np.stack(
        [np.interp(tgt_pos, src_pos, arr[t]) for t in range(T)],
        axis=0,
    )
    return torch.tensor(out, dtype=torch.float32)


In [ ]:
from torch.utils.data import Dataset


class MooringSequenceDatasetPositionTension(Dataset):
    """
    Short-history-window dataset for the mooring-line GAT-LSTM.

    Inputs are absolute node positions read directly from the .dat FE output
    (x_abs, z_abs, tension).  No displacement-to-position reconstruction.

    Dynamic node features (base 6, plus optional 7 env = 13 total):
    ---------------------------------------------------------------
    0  x_abs             absolute horizontal node position [m]
    1  z_abs             absolute vertical node position [m]
    2  tension           line tension [N]
    3  contact_flag      binary: 1 if node touches seabed
    4  penetration_depth vertical penetration below seabed [m]
    5  bed_reaction_z    vertical seabed reaction force [N]
    6+ Hs, Tp, cd1-cd5  environmental features (optional)

    Target features (3):
    --------------------
    0  x_abs
    1  z_abs
    2  tension
    """

    def __init__(
        self,
        graph_data,
        x_abs: torch.Tensor,
        z_abs: torch.Tensor,
        tension: torch.Tensor,
        history_len: int,
        env_features: torch.Tensor = None,
        future_len: int = 1,
        contact_tol: float = 1e-6,
    ):
        """
        Parameters
        ----------
        graph_data : torch_geometric.data.Data
            Static graph built by build_mooring_graph().
        x_abs : torch.Tensor  shape [T, N]
            Absolute horizontal node positions from .dat file [m].
        z_abs : torch.Tensor  shape [T, N]
            Absolute vertical node positions from .dat file [m].
        tension : torch.Tensor  shape [T, N]
            Line tension at each node [N].
        history_len : int
        env_features : torch.Tensor, optional  shape [T, n_env]
            Environmental features (Hs, Tp, cd1..cd5).
        future_len : int
        contact_tol : float
            Tolerance for seabed contact detection [m].
        """

        self.graph_data = graph_data
        self.history_len = history_len
        self.future_len = future_len

        # ------------------------------------------------------------------
        # 1. SHAPE CHECKS
        # ------------------------------------------------------------------

        expected_shape = x_abs.shape
        self.num_steps, self.num_nodes = expected_shape   # must be before env check

        for name, tensor in {"z_abs": z_abs, "tension": tension}.items():
            if tensor.shape != expected_shape:
                raise ValueError(
                    f"{name} has shape {tensor.shape}, expected {expected_shape}."
                )

        if env_features is not None:
            if env_features.ndim != 2 or env_features.shape[0] != self.num_steps:
                raise ValueError(
                    f"env_features must have shape [T={self.num_steps}, n_env], "
                    f"got {tuple(env_features.shape)}."
                )
        self.env_features = env_features

        if self.graph_data.num_nodes_total != self.num_nodes:
            raise ValueError(
                f"Graph has {self.graph_data.num_nodes_total} nodes, "
                f"dynamic data has {self.num_nodes} nodes."
            )

        if self.num_steps < history_len + future_len:
            raise ValueError(
                "Not enough time steps for history_len and future_len."
            )

        # ------------------------------------------------------------------
        # 2. SEABED INTERACTION FEATURES
        # ------------------------------------------------------------------

        # z_abs comes directly from the .dat file — no reconstruction needed.
        z_bed_node = self.graph_data.z_bed_node.unsqueeze(0)   # [1, N]

        contact_flag      = (z_abs <= z_bed_node + contact_tol).float()
        penetration_depth = torch.clamp(z_bed_node - z_abs, min=0.0)

        # Seabed reaction: k_b * penetration (damping omitted — no velocity input)
        bed_reaction_z = self.graph_data.k_b * penetration_depth
        bed_reaction_z = torch.where(
            contact_flag > 0.0, bed_reaction_z, torch.zeros_like(bed_reaction_z)
        )
        bed_reaction_z = torch.clamp(bed_reaction_z, min=0.0)

        # ------------------------------------------------------------------
        # 2b. DYNAMIC NODE FEATURES  [T, N, F]
        # Index layout:
        #   0  x_abs             continuous
        #   1  z_abs             continuous
        #   2  tension           continuous
        #   3  contact_flag      binary  (skip when normalising)
        #   4  penetration_depth continuous
        #   5  bed_reaction_z    continuous
        #   6+ env features      continuous (optional)
        # ------------------------------------------------------------------

        self.dynamic_features = torch.stack(
            [
                x_abs,             # 0
                z_abs,             # 1
                tension,           # 2
                contact_flag,      # 3
                penetration_depth, # 4
                bed_reaction_z,    # 5
            ],
            dim=-1,
        )   # [T, N, 6]

        if env_features is not None:
            n_env = env_features.shape[1]
            env_expanded = env_features.unsqueeze(1).expand(-1, self.num_nodes, n_env)
            self.dynamic_features = torch.cat(
                [self.dynamic_features, env_expanded], dim=-1
            )   # [T, N, 6 + n_env]
            env_names = ENV_FEATURE_NAMES[:n_env]
        else:
            env_names = []

        self.dynamic_feature_names = [
            "x_abs", "z_abs", "tension", "contact_flag",
            "penetration_depth", "bed_reaction_z",
        ] + env_names

        # ------------------------------------------------------------------
        # 2c. DYNAMIC EDGE FEATURES  [T, E, 4]
        # ------------------------------------------------------------------

        source_nodes = self.graph_data.edge_index[0]
        target_nodes = self.graph_data.edge_index[1]

        contact_src      = contact_flag[:, source_nodes]
        contact_tgt      = contact_flag[:, target_nodes]
        penetration_src  = penetration_depth[:, source_nodes]
        penetration_tgt  = penetration_depth[:, target_nodes]
        bed_reaction_src = bed_reaction_z[:, source_nodes]
        bed_reaction_tgt = bed_reaction_z[:, target_nodes]

        edge_contact_fraction  = 0.5 * (contact_src + contact_tgt)
        edge_penetration_mean  = 0.5 * (penetration_src + penetration_tgt)
        edge_bed_reaction_mean = 0.5 * (bed_reaction_src + bed_reaction_tgt)
        edge_touchdown_flag    = (contact_src != contact_tgt).float()

        self.dynamic_edge_features = torch.stack(
            [
                edge_contact_fraction,    # 0
                edge_penetration_mean,    # 1
                edge_bed_reaction_mean,   # 2
                edge_touchdown_flag,      # 3
            ],
            dim=-1,
        )   # [T, E, 4]

        self.dynamic_edge_feature_names = [
            "edge_contact_fraction",
            "edge_penetration_mean",
            "edge_bed_reaction_mean",
            "edge_touchdown_flag",
        ]

        # ------------------------------------------------------------------
        # 3. TARGETS  [T, N, 3]
        # ------------------------------------------------------------------

        self.targets = torch.stack([x_abs, z_abs, tension], dim=-1)
        self.target_feature_names = ["x_abs", "z_abs", "tension"]

        # ------------------------------------------------------------------
        # 4. WINDOW COUNT
        # ------------------------------------------------------------------

        self.num_windows = self.num_steps - self.history_len - self.future_len + 1

    def __len__(self):
        return self.num_windows

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.num_windows:
            raise IndexError("Sample index out of range.")

        input_start  = idx
        input_end    = idx + self.history_len
        target_start = input_end
        target_end   = target_start + self.future_len

        x_seq    = self.dynamic_features[input_start:input_end]        # [H, N, F]
        edge_seq = self.dynamic_edge_features[input_start:input_end]   # [H, E, 4]
        y_seq    = self.targets[target_start:target_end]               # [P, N, 3]

        return {
            "graph":     self.graph_data,
            "x_seq":     x_seq,
            "edge_seq":  edge_seq,
            "y_seq":     y_seq,
            "start_idx": idx,
        }


def print_dataset_summary(dataset: MooringSequenceDatasetPositionTension):
    print("----- DATASET SUMMARY -----")
    print(f"Time steps : {dataset.num_steps}")
    print(f"Nodes      : {dataset.num_nodes}")
    print(f"History len: {dataset.history_len}")
    print(f"Future len : {dataset.future_len}")
    print(f"Windows    : {len(dataset)}")
    print("Dynamic input features :", dataset.dynamic_feature_names)
    print("Dynamic edge  features :", dataset.dynamic_edge_feature_names)
    print("Target features        :", dataset.target_feature_names)
    sample = dataset[0]
    print("x_seq   :", tuple(sample["x_seq"].shape))
    print("edge_seq:", tuple(sample["edge_seq"].shape))
    print("y_seq   :", tuple(sample["y_seq"].shape))


In [3]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv


class MooringGATEncoder(nn.Module):
    """
    Spatial graph encoder for one history time step.

    It takes:
      - static node features from graph.x
      - static edge features from graph.edge_attr
      - dynamic node features for one time step x_t
      - dynamic edge features for one time step edge_t

    and returns:
      - node embeddings h_t for that time step

    Expected shapes for one sample:
      graph.x         : [N, n_static_node_features]
      graph.edge_index: [2, E]
      graph.edge_attr : [E, n_static_edge_features]
      x_t             : [N, n_dynamic_node_features]
      edge_t          : [E, n_dynamic_edge_features]

    Output:
      h_t             : [N, gat_hidden_dim]
    """

    def __init__(
        self,
        n_static_node_features: int = 13,
        n_dynamic_node_features: int = 10,
        n_static_edge_features: int = 8,
        n_dynamic_edge_features: int = 4,
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        dropout: float = 0.1,
        use_layernorm: bool = True,
        add_residual_projection: bool = True,
    ):
        super().__init__()

        self.n_static_node_features = n_static_node_features
        self.n_dynamic_node_features = n_dynamic_node_features
        self.n_static_edge_features = n_static_edge_features
        self.n_dynamic_edge_features = n_dynamic_edge_features

        self.node_input_dim = n_static_node_features + n_dynamic_node_features
        self.edge_input_dim = n_static_edge_features + n_dynamic_edge_features

        self.gat_hidden_dim = gat_hidden_dim
        self.gat_out_dim = gat_out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.use_layernorm = use_layernorm

        # -------------------------------------------------------------
        # GAT layer 1
        # concat=True => output dim = gat_hidden_dim * num_heads
        # -------------------------------------------------------------
        self.gat1 = GATv2Conv(
            in_channels=self.node_input_dim,
            out_channels=gat_hidden_dim,
            heads=num_heads,
            concat=True,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm1 = nn.LayerNorm(gat_hidden_dim * num_heads) if use_layernorm else nn.Identity()

        # -------------------------------------------------------------
        # GAT layer 2
        # concat=False => output dim = gat_out_dim
        # -------------------------------------------------------------
        self.gat2 = GATv2Conv(
            in_channels=gat_hidden_dim * num_heads,
            out_channels=gat_out_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm2 = nn.LayerNorm(gat_out_dim) if use_layernorm else nn.Identity()

        self.act = nn.ELU()
        self.dropout_layer = nn.Dropout(dropout)

        # Optional residual projection from raw concatenated node input
        if add_residual_projection:
            self.residual_proj = nn.Linear(self.node_input_dim, gat_out_dim)
        else:
            self.residual_proj = None

    def forward(self, graph, x_t: torch.Tensor, edge_t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object containing graph.x, graph.edge_index, graph.edge_attr.
        x_t : torch.Tensor
            Dynamic node features at one time step, shape [N, n_dynamic_node_features].
        edge_t : torch.Tensor
            Dynamic edge features at one time step, shape [E, n_dynamic_edge_features].

        Returns
        -------
        h_t : torch.Tensor
            Encoded node embeddings for this time step, shape [N, gat_out_dim].
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_t.dim() != 2:
            raise ValueError(f"x_t must have shape [N, F_dyn_node], got {tuple(x_t.shape)}")

        if edge_t.dim() != 2:
            raise ValueError(f"edge_t must have shape [E, F_dyn_edge], got {tuple(edge_t.shape)}")

        x_static = graph.x
        edge_index = graph.edge_index
        edge_static = graph.edge_attr

        if x_static.size(0) != x_t.size(0):
            raise ValueError(
                f"Node count mismatch: graph.x has {x_static.size(0)} nodes, "
                f"but x_t has {x_t.size(0)} nodes."
            )

        if edge_static.size(0) != edge_t.size(0):
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {edge_static.size(0)} edges, "
                f"but edge_t has {edge_t.size(0)} edges."
            )

        if x_static.size(1) != self.n_static_node_features:
            raise ValueError(
                f"Expected {self.n_static_node_features} static node features, "
                f"got {x_static.size(1)}."
            )

        if x_t.size(1) != self.n_dynamic_node_features:
            raise ValueError(
                f"Expected {self.n_dynamic_node_features} dynamic node features, "
                f"got {x_t.size(1)}."
            )

        if edge_static.size(1) != self.n_static_edge_features:
            raise ValueError(
                f"Expected {self.n_static_edge_features} static edge features, "
                f"got {edge_static.size(1)}."
            )

        if edge_t.size(1) != self.n_dynamic_edge_features:
            raise ValueError(
                f"Expected {self.n_dynamic_edge_features} dynamic edge features, "
                f"got {edge_t.size(1)}."
            )

        # -------------------------------------------------------------
        # Concatenate static + dynamic features for this time step
        # -------------------------------------------------------------
        x_in = torch.cat([x_static, x_t], dim=-1)           # [N, 13 + 10] = [N, 23]
        edge_in = torch.cat([edge_static, edge_t], dim=-1) # [E, 8 + 4]  = [E, 12]

        # -------------------------------------------------------------
        # GAT block 1
        # -------------------------------------------------------------
        h = self.gat1(x_in, edge_index, edge_in)            # [N, gat_hidden_dim * num_heads]
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout_layer(h)

        # -------------------------------------------------------------
        # GAT block 2
        # -------------------------------------------------------------
        h = self.gat2(h, edge_index, edge_in)               # [N, gat_out_dim]
        h = self.norm2(h)

        # -------------------------------------------------------------
        # Optional residual connection from raw input
        # -------------------------------------------------------------
        if self.residual_proj is not None:
            h = h + self.residual_proj(x_in)

        h = self.act(h)
        h = self.dropout_layer(h)

        return h
    
# -------------------------------------------------------------
# Quick shape test on one dataset sample
# -------------------------------------------------------------
sample = dataset[0]

graph = sample["graph"] if isinstance(sample, dict) else sample[0]
x_seq = sample["x_seq"] if isinstance(sample, dict) else sample[1]
edge_seq = sample["edge_seq"] if isinstance(sample, dict) else sample[2]

print("graph.x shape      :", graph.x.shape)
print("graph.edge_attr    :", graph.edge_attr.shape)
print("x_seq shape        :", x_seq.shape)
print("edge_seq shape     :", edge_seq.shape)

encoder = MooringGATEncoder(
    n_static_node_features=13,
    n_dynamic_node_features=10,
    n_static_edge_features=8,
    n_dynamic_edge_features=4,
    gat_hidden_dim=64,
    gat_out_dim=64,
    num_heads=4,
    dropout=0.1,
)

x_t = x_seq[0]         # one history step: [N, 10]
edge_t = edge_seq[0]   # one history step: [E, 4]

with torch.no_grad():
    h_t = encoder(graph, x_t, edge_t)

print("x_t shape          :", x_t.shape)
print("edge_t shape       :", edge_t.shape)
print("h_t shape          :", h_t.shape)   # expected [N, 64]

graph.x shape      : torch.Size([7, 13])
graph.edge_attr    : torch.Size([12, 8])
x_seq shape        : torch.Size([10, 7, 10])
edge_seq shape     : torch.Size([10, 12, 4])
x_t shape          : torch.Size([7, 10])
edge_t shape       : torch.Size([12, 4])
h_t shape          : torch.Size([7, 64])


In [4]:
import torch
import torch.nn as nn


class NodeTemporalLSTM(nn.Module):
    """
    Temporal encoder that processes the sequence of spatial node embeddings
    produced by the MooringGATEncoder.

    Input:
        H : [history_len, N, gat_out_dim]

    Output:
        node_temporal : [N, lstm_hidden_dim]

    Interpretation:
        For each node i, we take its embedding sequence across time:
            H[:, i, :]  -> [history_len, gat_out_dim]
        and pass it through an LSTM.

    Notes:
        - This block is node-wise in time.
        - It does NOT mix nodes with each other.
        - Spatial coupling has already been handled by the GAT encoder.
    """

    def __init__(
        self,
        input_dim: int = 64,
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        dropout: float = 0.1,
        bidirectional: bool = False,
        use_layernorm: bool = True,
        use_last_timestep: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.lstm_hidden_dim = lstm_hidden_dim
        self.num_lstm_layers = num_lstm_layers
        self.bidirectional = bidirectional
        self.use_layernorm = use_layernorm
        self.use_last_timestep = use_last_timestep

        # PyTorch LSTM only uses dropout internally when num_layers > 1
        lstm_dropout = dropout if num_lstm_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,   # input will be [N, history_len, input_dim]
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        self.output_dim = lstm_hidden_dim * (2 if bidirectional else 1)

        self.norm = nn.LayerNorm(self.output_dim) if use_layernorm else nn.Identity()
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        H : torch.Tensor
            Sequence of node embeddings from the spatial encoder.
            Expected shape: [history_len, N, input_dim]

        Returns
        -------
        node_temporal : torch.Tensor
            Temporal embedding for each node.
            Shape: [N, output_dim]
        """

        if H.dim() != 3:
            raise ValueError(
                f"H must have shape [history_len, N, input_dim], got {tuple(H.shape)}"
            )

        history_len, N, F = H.shape

        if F != self.input_dim:
            raise ValueError(
                f"Expected input_dim={self.input_dim}, but got last dimension {F}."
            )

        # Rearrange so each node becomes one sequence sample:
        # [history_len, N, input_dim] -> [N, history_len, input_dim]
        H_nodes = H.permute(1, 0, 2).contiguous()

        # LSTM output:
        #   lstm_out : [N, history_len, output_dim]
        #   h_n      : [num_layers * num_directions, N, lstm_hidden_dim]
        lstm_out, (h_n, c_n) = self.lstm(H_nodes)

        if self.use_last_timestep:
            # Use the output at the final history step
            node_temporal = lstm_out[:, -1, :]   # [N, output_dim]
        else:
            # Alternative: use the final hidden state
            if self.bidirectional:
                # last layer forward + last layer backward
                h_forward = h_n[-2]   # [N, lstm_hidden_dim]
                h_backward = h_n[-1]  # [N, lstm_hidden_dim]
                node_temporal = torch.cat([h_forward, h_backward], dim=-1)
            else:
                node_temporal = h_n[-1]  # [N, lstm_hidden_dim]

        node_temporal = self.norm(node_temporal)
        node_temporal = self.dropout_layer(node_temporal)

        return node_temporal
    

    # -------------------------------------------------------------
# Quick shape test for NodeTemporalLSTM
# -------------------------------------------------------------
sample = dataset[0]

graph = sample["graph"] if isinstance(sample, dict) else sample[0]
x_seq = sample["x_seq"] if isinstance(sample, dict) else sample[1]
edge_seq = sample["edge_seq"] if isinstance(sample, dict) else sample[2]

encoder = MooringGATEncoder(
    n_static_node_features=13,
    n_dynamic_node_features=10,
    n_static_edge_features=8,
    n_dynamic_edge_features=4,
    gat_hidden_dim=64,
    gat_out_dim=64,
    num_heads=4,
    dropout=0.1,
)

# Build spatial embeddings over the whole history window
h_list = []
with torch.no_grad():
    for t in range(x_seq.shape[0]):
        x_t = x_seq[t]         # [N, 10]
        edge_t = edge_seq[t]   # [E, 4]
        h_t = encoder(graph, x_t, edge_t)   # [N, 64]
        h_list.append(h_t)

H = torch.stack(h_list, dim=0)   # [history_len, N, 64]

temporal_block = NodeTemporalLSTM(
    input_dim=64,
    lstm_hidden_dim=128,
    num_lstm_layers=1,
    dropout=0.1,
    bidirectional=False,
    use_layernorm=True,
    use_last_timestep=True,
)

with torch.no_grad():
    node_temporal = temporal_block(H)

print("H shape               :", H.shape)               # [history_len, N, 64]
print("node_temporal shape   :", node_temporal.shape)   # [N, 128]

H shape               : torch.Size([10, 7, 64])
node_temporal shape   : torch.Size([7, 128])


In [5]:
import torch
import torch.nn as nn


class MooringGATLSTM(nn.Module):
    """
    Full spatiotemporal model for the mooring-line problem.

    Pipeline:
        1) For each history time step t:
              - combine static + dynamic graph information
              - run MooringGATEncoder
              - get node embeddings h_t

        2) Stack all h_t over the history window:
              H = [history_len, N, gat_out_dim]

        3) Run NodeTemporalLSTM on H:
              node_temporal = [N, temporal_dim]

        4) Predict all future steps directly for each node:
              y_hat = [future_len, N, output_dim]

    Default output_dim=3 corresponds to:
        [x_abs, z_abs, tension]
    """

    def __init__(
        self,
        # ----- graph feature sizes -----
        n_static_node_features: int = 13,
        n_dynamic_node_features: int = 6,
        n_static_edge_features: int = 8,
        n_dynamic_edge_features: int = 4,

        # ----- spatial encoder -----
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        gat_dropout: float = 0.1,
        gat_use_layernorm: bool = True,
        add_residual_projection: bool = True,

        # ----- temporal encoder -----
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        lstm_dropout: float = 0.1,
        bidirectional: bool = False,
        lstm_use_layernorm: bool = True,
        use_last_timestep: bool = True,

        # ----- prediction head -----
        future_len: int = 1,
        output_dim: int = 3,
        head_hidden_dim: int = 128,
        head_dropout: float = 0.1,
    ):
        super().__init__()

        self.future_len = future_len
        self.output_dim = output_dim

        # -------------------------------------------------------------
        # Spatial encoder: one time step -> node embeddings
        # -------------------------------------------------------------
        self.spatial_encoder = MooringGATEncoder(
            n_static_node_features=n_static_node_features,
            n_dynamic_node_features=n_dynamic_node_features,
            n_static_edge_features=n_static_edge_features,
            n_dynamic_edge_features=n_dynamic_edge_features,
            gat_hidden_dim=gat_hidden_dim,
            gat_out_dim=gat_out_dim,
            num_heads=num_heads,
            dropout=gat_dropout,
            use_layernorm=gat_use_layernorm,
            add_residual_projection=add_residual_projection,
        )

        # -------------------------------------------------------------
        # Temporal encoder: sequence of node embeddings -> one temporal
        # embedding per node
        # -------------------------------------------------------------
        self.temporal_encoder = NodeTemporalLSTM(
            input_dim=gat_out_dim,
            lstm_hidden_dim=lstm_hidden_dim,
            num_lstm_layers=num_lstm_layers,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
            use_layernorm=lstm_use_layernorm,
            use_last_timestep=use_last_timestep,
        )

        temporal_out_dim = self.temporal_encoder.output_dim

        # -------------------------------------------------------------
        # Prediction head:
        # [N, temporal_out_dim] -> [N, future_len * output_dim]
        # then reshape to [future_len, N, output_dim]
        # -------------------------------------------------------------
        self.prediction_head = nn.Sequential(
            nn.Linear(temporal_out_dim, head_hidden_dim),
            nn.ELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden_dim, future_len * output_dim),
        )

    def forward(self, graph, x_seq: torch.Tensor, edge_seq: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object.
        x_seq : torch.Tensor
            Dynamic node features over the history window.
            Expected shape: [history_len, N, n_dynamic_node_features]
        edge_seq : torch.Tensor
            Dynamic edge features over the history window.
            Expected shape: [history_len, E, n_dynamic_edge_features]

        Returns
        -------
        y_hat : torch.Tensor
            Predicted future targets.
            Shape: [future_len, N, output_dim]
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_seq.dim() != 3:
            raise ValueError(
                f"x_seq must have shape [history_len, N, F_dyn_node], got {tuple(x_seq.shape)}"
            )

        if edge_seq.dim() != 3:
            raise ValueError(
                f"edge_seq must have shape [history_len, E, F_dyn_edge], got {tuple(edge_seq.shape)}"
            )

        history_len_x, N_x, _ = x_seq.shape
        history_len_e, E_x, _ = edge_seq.shape

        if history_len_x != history_len_e:
            raise ValueError(
                f"History length mismatch: x_seq has {history_len_x}, edge_seq has {history_len_e}."
            )

        if graph.x.size(0) != N_x:
            raise ValueError(
                f"Node count mismatch: graph.x has {graph.x.size(0)} nodes, but x_seq has {N_x}."
            )

        if graph.edge_attr.size(0) != E_x:
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {graph.edge_attr.size(0)} edges, "
                f"but edge_seq has {E_x}."
            )

        # -------------------------------------------------------------
        # Spatial encoding over all history steps
        # -------------------------------------------------------------
        h_list = []

        for t in range(history_len_x):
            x_t = x_seq[t]         # [N, F_dyn_node]
            edge_t = edge_seq[t]   # [E, F_dyn_edge]

            h_t = self.spatial_encoder(graph, x_t, edge_t)   # [N, gat_out_dim]
            h_list.append(h_t)

        # Stack over time:
        # H = [history_len, N, gat_out_dim]
        H = torch.stack(h_list, dim=0)

        # -------------------------------------------------------------
        # Temporal encoding
        # -------------------------------------------------------------
        node_temporal = self.temporal_encoder(H)   # [N, temporal_out_dim]

        # -------------------------------------------------------------
        # Predict all future steps directly
        # -------------------------------------------------------------
        y_hat_flat = self.prediction_head(node_temporal)  # [N, future_len * output_dim]

        # reshape to [N, future_len, output_dim]
        y_hat = y_hat_flat.view(N_x, self.future_len, self.output_dim)

        # permute to match dataset target convention: [future_len, N, output_dim]
        y_hat = y_hat.permute(1, 0, 2).contiguous()

        return y_hat
    
# -------------------------------------------------------------
# Quick shape test for the full MooringGATLSTM
# -------------------------------------------------------------
sample = dataset[0]

graph = sample["graph"] if isinstance(sample, dict) else sample[0]
x_seq = sample["x_seq"] if isinstance(sample, dict) else sample[1]
edge_seq = sample["edge_seq"] if isinstance(sample, dict) else sample[2]
y_seq = sample["y_seq"] if isinstance(sample, dict) else sample[3]

print("graph.x shape       :", graph.x.shape)
print("graph.edge_attr     :", graph.edge_attr.shape)
print("x_seq shape         :", x_seq.shape)
print("edge_seq shape      :", edge_seq.shape)
print("y_seq shape         :", y_seq.shape)

future_len = y_seq.shape[0]
output_dim = y_seq.shape[-1]

model = MooringGATLSTM(
    n_static_node_features=13,
    n_dynamic_node_features=x_seq.shape[-1],
    n_static_edge_features=8,
    n_dynamic_edge_features=4,
    gat_hidden_dim=64,
    gat_out_dim=64,
    num_heads=4,
    gat_dropout=0.1,
    gat_use_layernorm=True,
    add_residual_projection=True,
    lstm_hidden_dim=128,
    num_lstm_layers=1,
    lstm_dropout=0.1,
    bidirectional=False,
    lstm_use_layernorm=True,
    use_last_timestep=True,
    future_len=future_len,
    output_dim=output_dim,
    head_hidden_dim=128,
    head_dropout=0.1,
)

with torch.no_grad():
    y_hat = model(graph, x_seq, edge_seq)

print("y_hat shape         :", y_hat.shape)   # expected [future_len, N, output_dim]
print("target shape        :", y_seq.shape)

# -------------------------------------------------------------
# Optional loss test
# -------------------------------------------------------------
criterion = nn.MSELoss()

with torch.no_grad():
    y_hat = model(graph, x_seq, edge_seq)
    loss = criterion(y_hat, y_seq)

print("test loss:", loss.item())

graph.x shape       : torch.Size([7, 13])
graph.edge_attr     : torch.Size([12, 8])
x_seq shape         : torch.Size([10, 7, 10])
edge_seq shape      : torch.Size([10, 12, 4])
y_seq shape         : torch.Size([1, 7, 3])
y_hat shape         : torch.Size([1, 7, 3])
target shape        : torch.Size([1, 7, 3])
test loss: 330186.15625


In [ ]:
# -------------------------------------------------------------
# imports and training config
# -------------------------------------------------------------

import os
import math
import copy
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


@dataclass
class TrainingConfig:
    split_seed: int = 42
    seed_list: Tuple[int, ...] = (42, 100, 1234, 5, 999)

    # Locations available from batchRieke dataset (loc02 and loc12 not used)
    # Locs 1,3-9 used for training (time-based train/val/test split per case)
    # Locs 10,11 used as held-out location-level test set
    train_lc_ids:      Tuple[int, ...] = (1, 3, 4, 5, 6, 7, 8, 9)
    test_extra_lc_ids: Tuple[int, ...] = (10, 11)
    node_counts: Tuple[int, ...] = (4, 5, 6, 7, 8, 10, 12, 15, 18, 21)

    test_time_fraction: float = 0.30
    train_fraction_within_dev_blocks: float = 0.70
    raw_block_len: int = 64

    # optimization
    num_epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 1
    grad_clip_max_norm: float = 1.0

    # scheduler / stopping
    use_scheduler: bool = True
    scheduler_factor: float = 0.5
    scheduler_patience: int = 5
    early_stopping_patience: int = 15
    min_delta: float = 1e-5

    checkpoint_dir: str = "./checkpoints_gat_lstm"
    eps: float = 1e-8

    # Dynamic node feature indices — layout (6 base + 7 env = 13 total):
    #   0  x_abs             continuous
    #   1  z_abs             continuous
    #   2  tension           continuous
    #   3  contact_flag      binary  (excluded from normalisation)
    #   4  penetration_depth continuous
    #   5  bed_reaction_z    continuous
    #   6  Hs                continuous  (env)
    #   7  Tp                continuous
    #   8  cd1               continuous
    #   9  cd2               continuous
    #  10  cd3               continuous
    #  11  cd4               continuous
    #  12  cd5               continuous
    node_continuous_idx: Tuple[int, ...] = (0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12)
    node_binary_idx:     Tuple[int, ...] = (3,)

    edge_continuous_idx: Tuple[int, ...] = (0, 1, 2)
    edge_binary_idx:     Tuple[int, ...] = (3,)

    target_continuous_idx: Tuple[int, ...] = (0, 1, 2)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [7]:
# -------------------------------------------------------------
# split utilities
# -------------------------------------------------------------

def get_window_span(dataset: MooringSequenceDatasetPositionTension) -> int:
    return dataset.history_len + dataset.future_len

def valid_window_starts_inside_raw_interval(
    dataset: MooringSequenceDatasetPositionTension,
    raw_start: int,
    raw_end_exclusive: int,
) -> List[int]:
    span = dataset.history_len + dataset.future_len
    starts = []
    for s in range(len(dataset)):
        if s >= raw_start and (s + span) <= raw_end_exclusive:
            starts.append(s)
    return starts

def chunk_raw_time_range(
    raw_start: int,
    raw_end_exclusive: int,
    block_len: int,
) -> List[Tuple[int, int]]:
    blocks = []
    cur = raw_start
    while cur < raw_end_exclusive:
        nxt = min(cur + block_len, raw_end_exclusive)
        blocks.append((cur, nxt))
        cur = nxt
    return blocks

def build_leakage_aware_splits(
    load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
) -> Dict[str, List]:
    """
    Build train / val / test window index lists.

    Dataset keys are (lc_id, case_id, target_N) 3-tuples.

    Split rules:
    - lc_id in train_lc_ids:
        * last 30% raw time per case -> test
        * first 70% raw time per case -> development (block-shuffled train/val)
    - lc_id in test_extra_lc_ids:
        * all windows -> test  (held-out locations)

    Each element of the returned lists is ((lc_id, case_id, target_N), window_idx).
    """
    rng = random.Random(cfg.split_seed)
    split = {"train": [], "val": [], "test": []}

    for (lc_id, case_id, target_N), ds in load_case_datasets.items():
        num_steps = ds.num_steps

        if lc_id in cfg.train_lc_ids:
            test_raw_start = int(math.floor((1.0 - cfg.test_time_fraction) * num_steps))

            test_window_ids = valid_window_starts_inside_raw_interval(
                ds, raw_start=test_raw_start, raw_end_exclusive=num_steps,
            )
            split["test"].extend(
                ((lc_id, case_id, target_N), w) for w in test_window_ids
            )

            blocks = chunk_raw_time_range(
                raw_start=0, raw_end_exclusive=test_raw_start,
                block_len=cfg.raw_block_len,
            )
            valid_blocks = []
            for b_start, b_end in blocks:
                w_ids = valid_window_starts_inside_raw_interval(
                    ds, raw_start=b_start, raw_end_exclusive=b_end,
                )
                if w_ids:
                    valid_blocks.append((b_start, b_end, w_ids))

            rng.shuffle(valid_blocks)
            n_train = int(math.floor(cfg.train_fraction_within_dev_blocks * len(valid_blocks)))
            for _, _, w_ids in valid_blocks[:n_train]:
                split["train"].extend(((lc_id, case_id, target_N), w) for w in w_ids)
            for _, _, w_ids in valid_blocks[n_train:]:
                split["val"].extend(((lc_id, case_id, target_N), w) for w in w_ids)

        elif lc_id in cfg.test_extra_lc_ids:
            split["test"].extend(
                ((lc_id, case_id, target_N), w) for w in range(len(ds))
            )

        else:
            raise ValueError(f"LC {lc_id} is not assigned to any split rule.")

    return split

def print_split_summary(
    load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
    split_indices: Dict[str, List],
):
    """Print window counts per split, aggregated by (lc_id, target_N)."""
    print("----- SPLIT SUMMARY -----")
    for split_name, pairs in split_indices.items():
        # Aggregate by (lc_id, target_N) to keep output readable
        counts: Dict[Tuple, int] = {}
        for (lc_id, case_id, target_N), _ in pairs:
            key = (lc_id, target_N)
            counts[key] = counts.get(key, 0) + 1
        total = sum(counts.values())
        print(f"\n{split_name.upper()} total windows: {total}")
        for (lc_id, target_N) in sorted(counts):
            print(f"  LC {lc_id:02d} N={target_N:02d}: {counts[(lc_id, target_N)]} windows")


In [8]:
# -------------------------------------------------------------
# subset dataset
# -------------------------------------------------------------

class MultiLoadCaseWindowSubset(Dataset):
    """
    Thin wrapper around multiple MooringSequenceDatasetPositionTension objects.
    Keys in load_case_datasets are (lc_id, case_id, target_N) 3-tuples.
    """
    def __init__(
        self,
        load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
        index_pairs: List,
    ):
        self.load_case_datasets = load_case_datasets
        self.index_pairs = index_pairs

    def __len__(self):
        return len(self.index_pairs)

    def __getitem__(self, idx):
        key, window_idx = self.index_pairs[idx]
        lc_id, case_id, node_count = key
        sample = self.load_case_datasets[key][window_idx]

        return {
            "graph":      sample["graph"],
            "x_seq":      sample["x_seq"],
            "edge_seq":   sample["edge_seq"],
            "y_seq":      sample["y_seq"],
            "start_idx":  sample["start_idx"],
            "lc_id":      lc_id,
            "case_id":    case_id,
            "node_count": node_count,
        }

def single_item_collate(batch):
    if len(batch) != 1:
        raise ValueError("single_item_collate expects batch_size=1.")
    return batch[0]


In [9]:
# -------------------------------------------------------------
# normalization utilities
# -------------------------------------------------------------

class FeatureStandardizer:
    """
    Per-feature z-score standardization for selected feature columns.

    Works with tensors shaped:
    - node inputs:   [history_len, N, F]
    - edge inputs:   [history_len, E, F]
    - targets:       [future_len, N, F]
    """
    def __init__(self, feature_idx: Tuple[int, ...], eps: float = 1e-8):
        self.feature_idx = list(feature_idx)
        self.eps = eps
        self.mean = None
        self.std = None

    def fit_from_tensor_list(self, tensor_list: List[torch.Tensor]):
        """
        Aggregate across all dimensions except the last feature dimension.
        """
        selected = []
        for x in tensor_list:
            xs = x[..., self.feature_idx].reshape(-1, len(self.feature_idx))
            selected.append(xs)

        big = torch.cat(selected, dim=0)
        self.mean = big.mean(dim=0)
        self.std = big.std(dim=0, unbiased=False).clamp_min(self.eps)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = (x_sel - self.mean.to(x.device)) / self.std.to(x.device)
        return x

    def inverse_transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = x_sel * self.std.to(x.device) + self.mean.to(x.device)
        return x
    
def fit_normalizers_from_train_subset(
    train_subset: MultiLoadCaseWindowSubset,
    cfg: TrainingConfig,
):
    node_norm = FeatureStandardizer(cfg.node_continuous_idx, eps=cfg.eps)
    edge_norm = FeatureStandardizer(cfg.edge_continuous_idx, eps=cfg.eps)
    target_norm = FeatureStandardizer(cfg.target_continuous_idx, eps=cfg.eps)

    node_tensors = []
    edge_tensors = []
    target_tensors = []

    for i in range(len(train_subset)):
        sample = train_subset[i]
        node_tensors.append(sample["x_seq"])
        edge_tensors.append(sample["edge_seq"])
        target_tensors.append(sample["y_seq"])

    node_norm.fit_from_tensor_list(node_tensors)
    edge_norm.fit_from_tensor_list(edge_tensors)
    target_norm.fit_from_tensor_list(target_tensors)

    return node_norm, edge_norm, target_norm

class NormalizedSubset(Dataset):
    """
    Applies train-fitted normalization on the fly.
    """
    def __init__(
        self,
        base_subset: MultiLoadCaseWindowSubset,
        node_norm: FeatureStandardizer,
        edge_norm: FeatureStandardizer,
        target_norm: FeatureStandardizer,
    ):
        self.base_subset = base_subset
        self.node_norm = node_norm
        self.edge_norm = edge_norm
        self.target_norm = target_norm

    def __len__(self):
        return len(self.base_subset)

    def __getitem__(self, idx):
        sample = self.base_subset[idx]

        return {
            "graph": sample["graph"],
            "x_seq": self.node_norm.transform(sample["x_seq"]),
            "edge_seq": self.edge_norm.transform(sample["edge_seq"]),
            "y_seq": self.target_norm.transform(sample["y_seq"]),
            "start_idx": sample["start_idx"],
            "lc_id": sample["lc_id"],
        }

In [10]:
# -------------------------------------------------------------
# dataloader builder
# -------------------------------------------------------------

def build_dataloaders(
    load_case_datasets: Dict[int, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
):
    split_indices = build_leakage_aware_splits(load_case_datasets, cfg)
    print_split_summary(load_case_datasets, split_indices)

    train_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["train"])
    val_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["val"])
    test_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["test"])

    node_norm, edge_norm, target_norm = fit_normalizers_from_train_subset(train_raw, cfg)

    train_ds = NormalizedSubset(train_raw, node_norm, edge_norm, target_norm)
    val_ds = NormalizedSubset(val_raw, node_norm, edge_norm, target_norm)
    test_ds = NormalizedSubset(test_raw, node_norm, edge_norm, target_norm)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=single_item_collate,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=single_item_collate,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=single_item_collate,
    )

    norms = {
        "node_norm": node_norm,
        "edge_norm": edge_norm,
        "target_norm": target_norm,
    }

    return train_loader, val_loader, test_loader, norms, split_indices

In [11]:
# -------------------------------------------------------------
# loss and metrics
# -------------------------------------------------------------

class SequenceSmoothL1Loss(nn.Module):
    def __init__(self, beta: float = 1.0):
        super().__init__()
        self.loss_fn = nn.SmoothL1Loss(beta=beta)

    def forward(self, y_hat, y_true):
        return self.loss_fn(y_hat, y_true)
    
@torch.no_grad()
def compute_physical_metrics_per_target(
    y_hat_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    mape_eps: float = 1e-8,
):
    """
    Inputs:
        y_hat_norm, y_true_norm: [future_len, N, output_dim] in normalized space

    Returns:
        {
            "MAE":  {target_name: value, ...},
            "RMSE": {target_name: value, ...},
            "MAPE": {"tension": value}   # only if tension exists
            "R2":   {target_name: value, ...},
        }
    """
    y_hat_phys = target_norm.inverse_transform(y_hat_norm.detach().cpu())
    y_true_phys = target_norm.inverse_transform(y_true_norm.detach().cpu())

    error = y_hat_phys - y_true_phys

    # MAE
    mae = error.abs().mean(dim=(0, 1))

    # RMSE
    rmse = torch.sqrt((error ** 2).mean(dim=(0, 1)))

    # R^2
    y_true_mean = y_true_phys.mean(dim=(0, 1), keepdim=True)
    ss_res = (error ** 2).sum(dim=(0, 1))
    ss_tot = ((y_true_phys - y_true_mean) ** 2).sum(dim=(0, 1))
    r2 = 1.0 - ss_res / ss_tot.clamp_min(mape_eps)

    metrics = {
        "MAE":  {name: float(mae[i].item()) for i, name in enumerate(target_names)},
        "RMSE": {name: float(rmse[i].item()) for i, name in enumerate(target_names)},
        "MAPE": {},
        "R2":   {name: float(r2[i].item()) for i, name in enumerate(target_names)},
    }

    # MAPE only for tension
    if "tension" in target_names:
        tension_idx = target_names.index("tension")
        denom = y_true_phys[..., tension_idx].abs().clamp_min(mape_eps)
        tension_mape = (error[..., tension_idx].abs() / denom).mean() * 100.0
        metrics["MAPE"]["tension"] = float(tension_mape.item())

    return metrics

In [12]:
# -------------------------------------------------------------
# model/optimizer setup
# -------------------------------------------------------------

def build_model_from_dataset_example(
    example_dataset: MooringSequenceDatasetPositionTension,
    device: torch.device,
):
    model = MooringGATLSTM(
        n_static_node_features=example_dataset.graph_data.x.shape[1],
        n_dynamic_node_features=example_dataset.dynamic_features.shape[-1],
        n_static_edge_features=example_dataset.graph_data.edge_attr.shape[1],
        n_dynamic_edge_features=example_dataset.dynamic_edge_features.shape[-1],
        future_len=example_dataset.future_len,
        output_dim=example_dataset.targets.shape[-1],

        # baseline hyperparameters
        gat_hidden_dim=64,
        gat_out_dim=64,
        num_heads=4,
        gat_dropout=0.1,
        gat_use_layernorm=True,
        add_residual_projection=True,

        lstm_hidden_dim=128,
        num_lstm_layers=1,
        lstm_dropout=0.1,
        bidirectional=False,
        lstm_use_layernorm=True,
        use_last_timestep=True,

        head_hidden_dim=128,
        head_dropout=0.1,
    ).to(device)

    return model

def build_optimizer_and_scheduler(
    model: nn.Module,
    cfg: TrainingConfig,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = None
    if cfg.use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
        )

    return optimizer, scheduler

In [13]:
# -------------------------------------------------------------
# training and validation loops
# -------------------------------------------------------------


def move_sample_to_device(sample, device):
    return {
        "graph": sample["graph"].to(device),
        "x_seq": sample["x_seq"].to(device),
        "edge_seq": sample["edge_seq"].to(device),
        "y_seq": sample["y_seq"].to(device),
        "start_idx": sample["start_idx"],
        "lc_id": sample["lc_id"],
    }

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    cfg: TrainingConfig,
):
    model.train()

    running_loss = 0.0
    num_batches = 0

    for sample in loader:
        sample = move_sample_to_device(sample, device)

        optimizer.zero_grad()

        y_hat = model(
            sample["graph"],
            sample["x_seq"],
            sample["edge_seq"],
        )

        loss = criterion(y_hat, sample["y_seq"])
        loss.backward()

        if cfg.grad_clip_max_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_max_norm)

        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    return running_loss / max(1, num_batches)

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
):
    model.eval()

    running_loss = 0.0
    num_batches = 0

    metric_sums = {
        "MAE":  {name: 0.0 for name in target_names},
        "RMSE": {name: 0.0 for name in target_names},
        "MAPE": {},
        "R2":   {name: 0.0 for name in target_names},
    }

    if "tension" in target_names:
        metric_sums["MAPE"]["tension"] = 0.0

    for sample in loader:
        sample = move_sample_to_device(sample, device)

        y_hat = model(
            sample["graph"],
            sample["x_seq"],
            sample["edge_seq"],
        )

        loss = criterion(y_hat, sample["y_seq"])
        running_loss += loss.item()
        num_batches += 1

        metrics_dict = compute_physical_metrics_per_target(
            y_hat_norm=y_hat,
            y_true_norm=sample["y_seq"],
            target_norm=target_norm,
            target_names=target_names,
        )

        for metric_name in ["MAE", "RMSE", "R2"]:
            for target_name in target_names:
                metric_sums[metric_name][target_name] += metrics_dict[metric_name][target_name]

        if "tension" in metrics_dict["MAPE"]:
            metric_sums["MAPE"]["tension"] += metrics_dict["MAPE"]["tension"]

    avg_loss = running_loss / max(1, num_batches)

    avg_metrics = {
        "MAE": {
            target_name: value / max(1, num_batches)
            for target_name, value in metric_sums["MAE"].items()
        },
        "RMSE": {
            target_name: value / max(1, num_batches)
            for target_name, value in metric_sums["RMSE"].items()
        },
        "MAPE": {},
        "R2": {
            target_name: value / max(1, num_batches)
            for target_name, value in metric_sums["R2"].items()
        },
    }

    if "tension" in metric_sums["MAPE"]:
        avg_metrics["MAPE"]["tension"] = metric_sums["MAPE"]["tension"] / max(1, num_batches)

    return avg_loss, avg_metrics

In [14]:
# -------------------------------------------------------------
# full training runner with checkpointing and early stopping
# -------------------------------------------------------------

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg: TrainingConfig,
    run_seed: int,
):
    checkpoint_dir = Path(cfg.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = checkpoint_dir / f"best_mooring_gat_lstm_seed_{run_seed}.pt"
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_metrics": [],
}

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, cfg.num_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            cfg=cfg,
        )

        val_loss, val_metrics = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            target_norm=target_norm,
            target_names=target_names,
        )

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_metrics"].append(val_metrics)

        mae_str = " | ".join([f"val_MAE_{k}={v:.6f}" for k, v in val_metrics["MAE"].items()])
        rmse_str = " | ".join([f"val_RMSE_{k}={v:.6f}" for k, v in val_metrics["RMSE"].items()])
        r2_str = " | ".join([f"val_R2_{k}={v:.6f}" for k, v in val_metrics["R2"].items()])

        extra_parts = [mae_str, rmse_str, r2_str]

        if "tension" in val_metrics["MAPE"]:
            extra_parts.append(f"val_MAPE_tension={val_metrics['MAPE']['tension']:.6f}")

        metrics_str = " | ".join(extra_parts)

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.6f} | "
            f"val_loss={val_loss:.6f} | "
            f"{metrics_str}"
)

        improved = (best_val_loss - val_loss) > cfg.min_delta
        if improved:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, str(best_model_path)

In [15]:
# -------------------------------------------------------------
# test evaluation
# -------------------------------------------------------------


@torch.no_grad()
def test_model(
    model,
    test_loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
):
    test_loss, test_metrics = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        target_norm=target_norm,
        target_names=target_names,
    )

    print("\n----- TEST RESULTS -----")
    print(f"test_loss = {test_loss:.6f}")

    for metric_name in ["MAE", "RMSE", "R2"]:
        for k, v in test_metrics[metric_name].items():
            print(f"test_{metric_name}_{k} = {v:.6f}")

    if "tension" in test_metrics["MAPE"]:
        print(f"test_MAPE_tension = {test_metrics['MAPE']['tension']:.6f}")

    return {
        "test_loss": test_loss,
        "test_metrics": test_metrics,
    }

In [16]:
def run_multi_seed_test_experiment(
    load_case_datasets: Dict[int, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
    device: torch.device,
):
    first_key = sorted(load_case_datasets.keys())[0]
    example_dataset = load_case_datasets[first_key]

    # Fixed split for all seeds
    train_loader, val_loader, test_loader, norms, split_indices = build_dataloaders(
        load_case_datasets=load_case_datasets,
        cfg=cfg,
    )

    all_results = []

    for run_seed in cfg.seed_list:
        print("\n" + "=" * 70)
        print(f"STARTING RUN FOR SEED {run_seed}")
        print("=" * 70)

        set_seed(run_seed)

        model = build_model_from_dataset_example(
            example_dataset=example_dataset,
            device=device,
        )

        criterion = SequenceSmoothL1Loss(beta=1.0)
        optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)

        model, history, model_path = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            criterion=criterion,
            device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names,
            cfg=cfg,
            run_seed=run_seed,
        )

        test_results = test_model(
            model=model,
            test_loader=test_loader,
            criterion=criterion,
            device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names,
        )

        run_result = {
            "seed": run_seed,
            "model_path": model_path,
            "history": history,
            "test_loss": test_results["test_loss"],
            "test_metrics": test_results["test_metrics"],
        }
        all_results.append(run_result)

    # Aggregate test loss across seeds
    test_losses = [r["test_loss"] for r in all_results]
    mean_test_loss = float(np.mean(test_losses))
    std_test_loss = float(np.std(test_losses))

    # Aggregate test across seeds
    target_names = example_dataset.target_feature_names
    metric_names_all_targets = ["MAE", "RMSE", "R2"]
    mean_test_metrics = {metric: {} for metric in metric_names_all_targets}
    std_test_metrics = {metric: {} for metric in metric_names_all_targets}
    mean_test_metrics["MAPE"] = {}
    std_test_metrics["MAPE"] = {}

    for metric in metric_names_all_targets:
        for name in target_names:
            vals = [r["test_metrics"][metric][name] for r in all_results]
            mean_test_metrics[metric][name] = float(np.mean(vals))
            std_test_metrics[metric][name] = float(np.std(vals))

    if "tension" in target_names:
        vals = [r["test_metrics"]["MAPE"]["tension"] for r in all_results]
        mean_test_metrics["MAPE"]["tension"] = float(np.mean(vals))
        std_test_metrics["MAPE"]["tension"] = float(np.std(vals))


    print("\n" + "=" * 70)
    print("MULTI-SEED TEST SUMMARY")
    print("=" * 70)
    print(f"Seeds used: {list(cfg.seed_list)}")
    print(f"Mean test loss across seeds = {mean_test_loss:.6f}")
    print(f"Std  test loss across seeds = {std_test_loss:.6f}")

    for metric in ["MAE", "RMSE", "R2"]:
        print(f"\n{metric}:")
        for name in target_names:
            print(
                f"  {name}: mean = {mean_test_metrics[metric][name]:.6f}, "
                f"std = {std_test_metrics[metric][name]:.6f}"
            )

    if "tension" in mean_test_metrics["MAPE"]:
        print("\nMAPE:")
        print(
            f"  tension: mean = {mean_test_metrics['MAPE']['tension']:.6f}, "
            f"std = {std_test_metrics['MAPE']['tension']:.6f}"
        )

    summary = {
        "all_results": all_results,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "mean_test_metrics": mean_test_metrics,
        "std_test_metrics": std_test_metrics,
    }
    return summary

In [ ]:
# -------------------------------------------------------------
# real data loader
# -------------------------------------------------------------

import os
import pandas as pd

DATA_ROOT         = r"C:\Users\thano\Desktop\data"
AVAILABLE_LOC_IDS = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]
N_CASES           = 300
N_NODES_FULL      = 21

# Set these to match the column names in your environmental CSV.
# Call inspect_env_csv(loc_id) first to check the actual column names.
ENV_COL_NAMES = ["Hs", "Tp", "cd1", "cd2", "cd3", "cd4", "cd5"]


def _loc_folder(loc_id: int) -> str:
    return os.path.join(DATA_ROOT, f"batchRieke_loc{loc_id:02d}")


def inspect_env_csv(loc_id: int) -> None:
    """Print column names and first row of the env CSV for loc_id."""
    input_dir = os.path.join(_loc_folder(loc_id), "input")
    csvs = [f for f in os.listdir(input_dir) if f.lower().endswith(".csv")]
    if not csvs:
        raise FileNotFoundError(f"No CSV found in {input_dir}")
    df = pd.read_csv(os.path.join(input_dir, csvs[0]))
    print(f"Loc {loc_id:02d} env CSV: {csvs[0]}")
    print(f"  Columns ({len(df.columns)}): {list(df.columns)}")
    print(f"  First row: {df.iloc[0].to_dict()}")
    print(f"  Shape: {df.shape}")


def load_env_csv(loc_id: int) -> pd.DataFrame:
    """Return the env DataFrame (300 rows x n_env cols) for a location."""
    input_dir = os.path.join(_loc_folder(loc_id), "input")
    csvs = [f for f in os.listdir(input_dir) if f.lower().endswith(".csv")]
    if len(csvs) != 1:
        raise FileNotFoundError(
            f"Expected 1 CSV in {input_dir}, found: {csvs}"
        )
    return pd.read_csv(os.path.join(input_dir, csvs[0]))


def case_env_tensor(env_df: pd.DataFrame, case_id: int, T: int) -> torch.Tensor:
    """
    Return a [T, 7] float32 tensor with constant env features for one case.

    Parameters
    ----------
    env_df  : DataFrame returned by load_env_csv().
    case_id : 1-indexed case number.
    T       : number of time steps (tensor rows will all be identical).
    """
    row_values = env_df.loc[case_id - 1, ENV_COL_NAMES].values.astype(np.float32)
    row_t = torch.tensor(row_values)               # [7]
    return row_t.unsqueeze(0).expand(T, -1).contiguous()   # [T, 7]


def parse_dat_file(dat_path: str) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Parse one gnl_data1.dat file.

    File format (no header, comma-separated):
        87 columns = 3 lead columns + 4 columns * 21 nodes

    Lead columns:
        0  time
        1  unused
        2  unused

    Per-node block (4 columns, repeated 21 times):
        0  reference  (dropped)
        1  x_abs      [m]
        2  z_abs      [m]
        3  tension    [N]

    Returns
    -------
    x_abs   : torch.Tensor  [T, 21]  float32
    z_abs   : torch.Tensor  [T, 21]  float32
    tension : torch.Tensor  [T, 21]  float32
    """
    raw_df = pd.read_csv(dat_path, sep=",", header=None)
    if raw_df.shape[1] != 87:
        raise ValueError(
            f"Expected 87 columns in {dat_path}, got {raw_df.shape[1]}."
        )

    # Drop columns 1 and 2 (unused); keep col 0 (time) + 84 node columns = 85 cols
    df = raw_df.drop(columns=[1, 2])

    # Node data: columns 1..84 (skip time at column 0)
    node_values = df.iloc[:, 1:].values          # [T, 84]  numpy float64
    T = node_values.shape[0]

    # Reshape to [T, 21, 4]: axis-2 = [reference, x_abs, z_abs, tension]
    node_data = node_values.reshape(T, N_NODES_FULL, 4)

    x_abs   = torch.tensor(node_data[:, :, 1], dtype=torch.float32)   # [T, 21]
    z_abs   = torch.tensor(node_data[:, :, 2], dtype=torch.float32)   # [T, 21]
    tension = torch.tensor(node_data[:, :, 3], dtype=torch.float32)   # [T, 21]

    return x_abs, z_abs, tension


In [ ]:
# -------------------------------------------------------------
# build load_case_datasets from real .dat files
# -------------------------------------------------------------
#
# Call inspect_env_csv(loc_id) first to verify ENV_COL_NAMES matches
# your CSV column headers.  Example:
#   inspect_env_csv(1)
#
# Memory note: 10 locs * 300 cases * 10 node counts = 30 000 datasets.
# For a quick test, reduce AVAILABLE_LOC_IDS or set a lower N_CASES_TO_LOAD.

N_CASES_TO_LOAD = N_CASES     # reduce to e.g. 10 for rapid testing
HISTORY_LEN     = 10
FUTURE_LEN      = 1
NODE_COUNTS     = [4, 5, 6, 7, 8, 10, 12, 15, 18, 21]

load_case_datasets = {}

for loc_id in AVAILABLE_LOC_IDS:
    print(f"\n--- Location {loc_id:02d} ---")
    env_df = load_env_csv(loc_id)

    for case_id in range(1, N_CASES_TO_LOAD + 1):
        dat_path = os.path.join(
            DATA_ROOT,
            f"batchRieke_loc{loc_id:02d}",
            f"case_{case_id:04d}",
            "gnl_data1.dat",
        )

        x_abs_full, z_abs_full, tension_full = parse_dat_file(dat_path)
        T = x_abs_full.shape[0]                    # typically 13 800
        env_t = case_env_tensor(env_df, case_id, T)

        for target_N in NODE_COUNTS:
            graph = build_mooring_graph(
                location_id=loc_id, num_intermediate=target_N - 2
            )

            x_abs   = resample_fe_output(x_abs_full,   target_N)
            z_abs   = resample_fe_output(z_abs_full,   target_N)
            tension = resample_fe_output(tension_full, target_N)

            load_case_datasets[(loc_id, case_id, target_N)] = \
                MooringSequenceDatasetPositionTension(
                    graph_data  = graph,
                    x_abs       = x_abs,
                    z_abs       = z_abs,
                    tension     = tension,
                    env_features= env_t,
                    history_len = HISTORY_LEN,
                    future_len  = FUTURE_LEN,
                    contact_tol = 1e-6,
                )

        if case_id % 50 == 0:
            print(f"  case {case_id:04d}/{N_CASES_TO_LOAD} loaded")

print(f"\nTotal datasets: {len(load_case_datasets):,}")
print(f"  = {len(AVAILABLE_LOC_IDS)} locs"
      f" x {N_CASES_TO_LOAD} cases"
      f" x {len(NODE_COUNTS)} node counts")

# ------------------------------------------------------------------
# Consistency checks
# ------------------------------------------------------------------
assert len(load_case_datasets) > 0, "load_case_datasets is empty."

first_key     = sorted(load_case_datasets.keys())[0]
example_ds    = load_case_datasets[first_key]

ref_dyn_node  = example_ds.dynamic_features.shape[-1]
ref_dyn_edge  = example_ds.dynamic_edge_features.shape[-1]
ref_target    = example_ds.targets.shape[-1]
ref_hist      = example_ds.history_len
ref_fut       = example_ds.future_len
ref_node_feat = example_ds.graph_data.x.shape[1]
ref_edge_feat = example_ds.graph_data.edge_attr.shape[1]

for (lc_id, case_id, target_N), ds in load_case_datasets.items():
    tag = f"({lc_id},{case_id},{target_N})"
    assert ds.dynamic_features.shape[-1]      == ref_dyn_node,  f"{tag}: dyn node feat mismatch"
    assert ds.dynamic_edge_features.shape[-1] == ref_dyn_edge,  f"{tag}: dyn edge feat mismatch"
    assert ds.targets.shape[-1]               == ref_target,    f"{tag}: target dim mismatch"
    assert ds.history_len                     == ref_hist,       f"{tag}: history_len mismatch"
    assert ds.future_len                      == ref_fut,        f"{tag}: future_len mismatch"
    assert ds.graph_data.x.shape[1]           == ref_node_feat, f"{tag}: static node feat mismatch"
    assert ds.graph_data.edge_attr.shape[1]   == ref_edge_feat, f"{tag}: static edge feat mismatch"
    assert ds.graph_data.num_nodes_total      == target_N,      f"{tag}: node count mismatch"

print("All consistency checks passed.")

# ------------------------------------------------------------------
# Run training
# ------------------------------------------------------------------
cfg    = TrainingConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

multi_seed_test_summary = run_multi_seed_test_experiment(
    load_case_datasets = load_case_datasets,
    cfg                = cfg,
    device             = device,
)


In [ ]:
# -------------------------------------------------------------
# inference example – variable node count at test/deployment time
# -------------------------------------------------------------

# Build a 5-node graph for location 4 (3 intermediate nodes)
graph_5 = build_mooring_graph(location_id=4, num_intermediate=3)

# x_seq_5:    [history_len, 5, F_dyn_node]  – from real sensor data or resampled FE output
# edge_seq_5: [history_len, 8, F_dyn_edge]  – 5 nodes -> 8 directed edges
#
# Retrieve feature dimensions from any training dataset
F_dyn_node = example_dataset.dynamic_features.shape[-1]
F_dyn_edge = example_dataset.dynamic_edge_features.shape[-1]

n_nodes_5 = graph_5.num_nodes_total          # 5
n_edges_5 = graph_5.edge_index.shape[1]      # 8 directed edges

# Replace zeros with real sensor readings or resample_fe_output output
x_seq_5    = torch.zeros(HISTORY_LEN, n_nodes_5, F_dyn_node)
edge_seq_5 = torch.zeros(HISTORY_LEN, n_edges_5, F_dyn_edge)

model.eval()
with torch.no_grad():
    y_hat = model(graph_5.to(device), x_seq_5.to(device), edge_seq_5.to(device))
# y_hat shape: [future_len, 5, 3]  ->  x_abs, z_abs, tension at each of the 5 node positions

print(f"Inference output shape: {y_hat.shape}")   # expected: [1, 5, 3]
print(y_hat)
